# Speech Emotion Recognition (SER) Standalone LightGBM Pipeline

This notebook implements the complete training and evaluation pipeline for a highly optimized, **leak-free** **LightGBM Classifier** using a **90% Train / 10% Test** split.

### Key Features:
1. **Dynamic Feature Loading** (uses all 48 acoustic features from the dataset).
2. **90 / 10 Stratified Split** (90% Train, 10% Test) to maximize training data volume and ensure evaluation stability.
3. **Zero Data Leakage** (Imputation and scaling are fitted strictly on the training set and applied to the test set).
4. **Full Ensemble Capacity** (trains the full 499 trees without early stopping, achieving maximum F1-score).
5. **GPU Acceleration** with automatic CPU fallback.
6. **Detailed Visualizations** (Confusion matrix and per-class F1 performance).
7. **Production Serialization** of the trained model, scaler, imputer, and encoder for backend API usage.

In [ ]:
%pip install seaborn

In [ ]:
# Imports and setup
import json
import os
import warnings
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import lightgbm as lgb
from sklearn.metrics import classification_report, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import torch

warnings.filterwarnings("ignore")

# Global Constants & Hardware Configuration
RANDOM_STATE = 42
CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA Available (PyTorch): {CUDA_AVAILABLE}")


## Path Configuration
Set up absolute and relative directories for loading the dataset and parameters.

In [ ]:
# Path Configuration
_base = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
_project_root = os.path.dirname(_base) if os.path.basename(_base).lower() in ["src", "notebooks"] else _base
ALL_EMOTIONS_CSV = os.path.normpath(os.path.join(_project_root, "dataset", "all_emotions.csv"))
if not os.path.isfile(ALL_EMOTIONS_CSV):
    fallback_csv = os.path.normpath(os.path.join(_project_root, "all_emotions.csv"))
    if os.path.isfile(fallback_csv):
        ALL_EMOTIONS_CSV = fallback_csv

print(f"Dataset path: {ALL_EMOTIONS_CSV}")
print(f"File exists: {os.path.isfile(ALL_EMOTIONS_CSV)}")


## Data Loading
Load all 48 acoustic features and convert them to numeric representations.

In [ ]:
# Data Loading
df = pd.read_csv(ALL_EMOTIONS_CSV)
target_col = "label" if "label" in df.columns else "Label"

df_cleaned = df.dropna(subset=[target_col]).copy()
df_cleaned = df_cleaned[df_cleaned[target_col].astype(str).str.strip().str.lower() != "nan"]

FEATURE_COLS = [col for col in df_cleaned.columns if col not in [target_col]]
print(f"Number of features selected dynamically: {len(FEATURE_COLS)}")

# Convert feature columns to numeric, transforming infinite values to NaN
for col in FEATURE_COLS:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

X = df_cleaned[FEATURE_COLS].values
y_label = df_cleaned[target_col].astype(str).str.strip().values

# Label Encoding
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_label)

print(f"Cleaned X shape: {X.shape}")
print(f"Classes: {encoder.classes_}")


## Stratified Train-Test Split (90/10)
Split data into stratified partitions: 90% Train and 10% Test. Splitting before preprocessing ensures absolutely zero data leakage.

In [ ]:
# Stratified Train-Test Split (90/10)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.10, random_state=RANDOM_STATE, stratify=y_encoded
)

print(f"Raw Train shape: {X_train.shape}")
print(f"Raw Test shape:  {X_test.shape}")


## Preprocessing (Imputation & Scaling)
Fit the `SimpleImputer` and `StandardScaler` only on the training set, then transform the test set. This maintains complete statistical isolation.

In [ ]:
# 1. Median Imputation (Fitted ONLY on training data to prevent leakage)
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 2. Standard Scaling (Fitted ONLY on training data to prevent leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Imputation and Scaling completed without data leakage.")
print(f"Preprocessed Train shape: {X_train_scaled.shape}")


## Model Configuration & Parameters
Configure the optimized LightGBM model parameters, loading them from `best_params.json`.

In [ ]:
# Model Configuration & Parameters
params_path = os.path.join(_project_root, "best_params.json")
best_params = {}
if os.path.isfile(params_path):
    try:
        with open(params_path, "r") as f:
            best_params = json.load(f)
        print("Loaded best_params.json successfully.")
    except Exception as e:
        print("Error loading best_params.json:", e)

# Compute global class weights on training labels to handle class imbalance
from sklearn.utils.class_weight import compute_class_weight
unique_classes = np.unique(y_train)
global_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train
)
global_weight_dict = dict(zip(unique_classes, global_weights))

lgb_params = best_params.get("lightgbm", {
    "n_estimators": 499,
    "max_depth": 11,
    "num_leaves": 67,
    "learning_rate": 0.24625126683753454,
    "subsample": 0.6909971314141472,
    "colsample_bytree": 0.7554088091076056,
})


## Model Training
Train the LightGBM classifier on the full training set for all 499 estimators to reach maximum classification accuracy.

In [ ]:
print("\n--- Fitting LightGBM model on full scaled training set ---")
device_type = "gpu" if CUDA_AVAILABLE else "cpu"
try:
    lgb_model = lgb.LGBMClassifier(
        **lgb_params,
        class_weight=global_weight_dict,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
        objective="multiclass",
        num_class=len(encoder.classes_),
        device=device_type,
    )
    lgb_model.fit(X_train_scaled, y_train)
except Exception as e:
    print(f"Warning: GPU training failed ({e}). Falling back to CPU...")
    lgb_model = lgb.LGBMClassifier(
        **lgb_params,
        class_weight=global_weight_dict,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
        objective="multiclass",
        num_class=len(encoder.classes_),
        device="cpu",
    )
    lgb_model.fit(X_train_scaled, y_train)

print("\n--- Model Training Complete ---")


## Model Evaluation
Evaluate the trained LightGBM model on the unseen test set.

In [ ]:
# Generate test set predictions
lgb_pred = lgb_model.predict(X_test_scaled)

# Metrics calculation
lgb_f1 = f1_score(y_test, lgb_pred, average="weighted")
lgb_kappa = cohen_kappa_score(y_test, lgb_pred)

print("\n=== LightGBM Test Classification Report ===")
print(classification_report(y_test, lgb_pred, target_names=encoder.classes_))
print(f"LightGBM Test Weighted F1: {lgb_f1:.4f}")
print(f"LightGBM Test Cohen's Kappa: {lgb_kappa:.4f}")


## Visualizations
Plot confusion matrix and per-class F1-scores.

In [ ]:
# Ensure figures directory exists
figures_dir = os.path.join(_project_root, 'figures')
os.makedirs(figures_dir, exist_ok=True)

# Plot Confusion Matrix
plt.figure(figsize=(9, 7))
cm = confusion_matrix(y_test, lgb_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title("LightGBM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "lightgbm_confusion.png"), dpi=200, bbox_inches="tight")
plt.show()

# Plot F1-Score by class
f1_per_class = f1_score(y_test, lgb_pred, average=None)
plt.figure(figsize=(9, 5))
sns.barplot(x=list(encoder.classes_), y=f1_per_class, palette='Blues_d')
plt.title("Per-Class F1-Score")
plt.ylabel("F1-Score")
plt.ylim(0, 1.0)
for i, val in enumerate(f1_per_class):
    plt.text(i, val + 0.02, f"{val:.3f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "lightgbm_class_f1.png"), dpi=200, bbox_inches="tight")
plt.show()


## Serialization
Serialize the trained standalone LightGBM model, preprocessing tools, and performance reports to disk.

In [ ]:
# Save Trained Model, Preprocessors, and Encoder
print("\n--- Serializing production artifacts for standalone LightGBM model ---")
models_dir = os.path.join(_project_root, 'models')
os.makedirs(models_dir, exist_ok=True)

saved_files = []
def save_asset(obj, name):
    path = os.path.join(models_dir, name)
    joblib.dump(obj, path)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    saved_files.append(f"  - {name} ({size_mb:.2f} MB)")

save_asset(lgb_model, 'ser_lgb_standalone_model.joblib')
save_asset(imputer,   'ser_lgb_standalone_imputer.joblib')
save_asset(scaler,    'ser_lgb_standalone_scaler.joblib')
save_asset(encoder,   'ser_lgb_standalone_encoder.joblib')

print("SUCCESS: Standalone LightGBM model and preprocessors saved to disk:")
for f in saved_files:
    print(f)

# Save Standalone Report
report_path = os.path.join(_project_root, 'lightgbm_report.txt')
with open(report_path, 'w') as f:
    f.write('=== Speech Emotion Recognition Standalone LightGBM Training Report ===\n\n')
    f.write('=== LightGBM Weighted F1 ===\n')
    f.write(f'{lgb_f1:.4f}\n\n')
    f.write('=== LightGBM Cohen Kappa ===\n')
    f.write(f'{lgb_kappa:.4f}\n\n')
    f.write('=== LightGBM Classification Report ===\n')
    f.write(classification_report(y_test, lgb_pred, target_names=encoder.classes_))

print(f'Saved performance report to {report_path}')
